# Tutorial 04: Holography Mask

This notebook focuses on the front aperture, also called the FTH holography mask.

The mask is a 3-D array with shape `(layers, y, x)`. Values close to `1` mean material is present. Values close to `0` mean the material has been opened by a hole. Object holes (`OH`) expose the magnetic sample; reference holes (`RH`) create reference waves for Fourier transform holography. Rectangular slits (`SLIT`) are also supported; they use the aperture radius list as slit width plus an `apertures_length` entry for the long side.

The sample-to-detector selector below defaults to the fast Fraunhofer FFT. See [Tutorial 16](16_compare_detector_propagation.ipynb) for the same-exit-wave Rayleigh–Sommerfeld comparison. This notebook stops before detector propagation or operates on supplied images; the selector is provided for extending the workflow. 


In [ ]:
# Sample-to-detector propagation (independent of multislice).
detector_propagation_method = "fraunhofer"  # Default; opt in with "rayleigh_sommerfeld".
# Direct Rayleigh-Sommerfeld is expensive: try small grids first.
# This tutorial does not propagate to a detector; pass this setting to
# DetectorConfig/HologramPipelineConfig when extending it to generate holograms.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the notebook runnable from a fresh clone without requiring an editable install.
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    # Use the interactive widget backend when it is available in JupyterLab.
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    # Plain scripts and some notebook renderers do not understand IPython magics.
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.sample_generator import pattern_generator


## 1. Simulation grid and layer thicknesses

The mask uses sample-plane pixels, not detector pixels. All aperture radii, centres, smoothing sigmas, and depths are entered in metres.


In [ ]:
sample_shape_2d = (1024, 1024)
real_space_pixel_size = 3e-9       # m / sample pixel

# A compact pedagogical mask stack: top absorber, membrane, magnetic film.
# The object hole will stop at the membrane, while reference holes go through all layers.
aperture_layer_names = ["Au", "SiN", "Co"]
aperture_thicknesses = [120e-9, 80e-9, 20e-9]
aperture_shape = (len(aperture_thicknesses), *sample_shape_2d)
thickness_OH = aperture_thicknesses[0] + aperture_thicknesses[1]


## 2. Define object and reference holes

Centres are `(y, x)` offsets from the sample centre in metres.

The top radius factor controls conical/tapered holes. A value of `2.0` means the top opening is twice the base radius.


In [ ]:
aperture_config = {
    "apertures_type": ["OH", "RH", "RH"],
    "apertures_radius": [555e-9, 100e-9, 34e-9],
    "apertures_center": [(0.0, 0.0), (-1240e-9, -1225e-9), (1225e-9, -1200e-9)],
    "apertures_sigma": [4e-9, 2e-9, 2e-9],
    "apertures_angle": [0.0, 0.0, 0.0],
    "apertures_ellipticity": [1.0, 1.0, 1.0],
    "apertures_roughness": [0.0, 0.02, 0.02],
    "apertures_roughness_modes": [(0, 0), (3, 10), (3, 10)],
    "apertures_seed": [1, 2, 3],
    "apertures_top_radius_factor": [1.3, 1.5, 1.75],
    "aperture_taper_depth": aperture_thicknesses[0],
    "thickness_OH": thickness_OH,
}

front_aperture_config = sim.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=aperture_shape,
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=aperture_thicknesses,
    aperture_layer_names=aperture_layer_names,
    aperture_config=aperture_config,
    use_roi=True,
)
front_aperture_config.setup()
aperture_mask = front_aperture_config.return_aperture()

print("3-D aperture mask shape:", aperture_mask.shape)
print("Mask value range:", aperture_mask.min(), aperture_mask.max())


## 3. Projected mask visualization

The depth average shows where the mask is open in projection. It is useful for a quick check, but it hides the vertical tapering of each hole.


In [ ]:
front_aperture_config.visualize_aperture()


## 4. Layer-by-layer view

This makes the depth dependence explicit. In this convention, darker pixels are more open.


In [ ]:
extent = 1e6 * front_aperture_config.aperture.get_illumination_extent_real_space()
n_layers = aperture_mask.shape[0]

fig, axes = plt.subplots(1, n_layers, figsize=(4 * n_layers, 3.5), sharex=True, sharey=True)
if n_layers == 1:
    axes = [axes]
for layer_index, ax in enumerate(axes):
    im = ax.imshow(aperture_mask[layer_index], extent=extent, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Layer {layer_index}")
    ax.set_xlabel("x in um")
    ax.set_ylabel("y in um")
fig.colorbar(im, ax=axes, label="material fraction")


## 5. Vertical cuts through each aperture

The cuts show whether an aperture is cylindrical, tapered, shallow, or fully drilled.


In [ ]:
centres_px = []
for centre_m in aperture_config["apertures_center"]:
    y_px = int(round(sample_shape_2d[0] / 2 + centre_m[0] / real_space_pixel_size))
    x_px = int(round(sample_shape_2d[1] / 2 + centre_m[1] / real_space_pixel_size))
    centres_px.append((y_px, x_px))

fig, axes = plt.subplots(len(centres_px), 2, figsize=(9, 3 * len(centres_px)))
for row, ((y_px, x_px), typ) in enumerate(zip(centres_px, aperture_config["apertures_type"])):
    yz = aperture_mask[:, :, x_px]
    xz = aperture_mask[:, y_px, :]
    axes[row, 0].imshow(yz, aspect="auto", cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"{typ}: y-z cut at x={x_px}")
    axes[row, 0].set_xlabel("y pixel")
    axes[row, 0].set_ylabel("layer")
    axes[row, 1].imshow(xz, aspect="auto", cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title(f"{typ}: x-z cut at y={y_px}")
    axes[row, 1].set_xlabel("x pixel")
    axes[row, 1].set_ylabel("layer")


## 6. Support masks for reconstruction

`create_supportmask()` can make masks on a different grid, for example the detector reconstruction grid. Use the type filter to isolate the object hole.


In [ ]:
all_holes = front_aperture_config.create_supportmask()
object_hole_only = front_aperture_config.create_supportmask(aperture_types=("OH",))
reference_holes_only = front_aperture_config.create_supportmask(aperture_types=("RH",))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharex=True, sharey=True)
for ax, data, title in zip(
    axes,
    [all_holes, object_hole_only, reference_holes_only],
    ["all holes", "object hole", "reference holes"],
):
    ax.imshow(data, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_axis_off()
